In [1]:
!tree ..

..
├── app.py
├── bfs_output.md
├── cleaned_output.md
├── config.py
├── config.yml
├── crawl_output.md
├── docs
│   └── raw_text_extraction.md
├── exceptions.py
├── experiments
│   ├── distilable_test.py
│   └── notebook1.ipynb
├── generate_jsonl_qapairs.py
├── main.py
├── output.md
├── pipeline
├── prompts
│   ├── __pycache__
│   │   ├── question_generation.cpython-310.pyc
│   │   └── system_prompt.cpython-310.pyc
│   ├── question_generation.py
│   └── system_prompt.py
├── __pycache__
│   ├── config.cpython-310.pyc
│   ├── exceptions.cpython-310.pyc
│   ├── main.cpython-310.pyc
│   ├── qa_generator.cpython-310.pyc
│   ├── qa_generator.cpython-312.pyc
│   └── utils.cpython-310.pyc
├── qa_generator.py
├── steps
│   └── raw_data_extraction.py
└── utils.py

8 directories, 26 files


In [2]:
import sys
import os

# Add the parent folder (project root) to sys.path
sys.path.append(os.path.abspath(".."))

# Now import works
from qa_generator import ProcessMarkdownQAPairs, text_splitter

/home/mindmap/Desktop/SLM/.venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/mindmap/Desktop/SLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("/home/mindmap/Desktop/SLM/unstructured_data/cleaned_output.md", "r") as f:
    data = f.read()

chunks = text_splitter(data)
len(chunks)

1083

In [4]:
obj = ProcessMarkdownQAPairs()
questions = obj.generate_questions(chunks[0])

2026-02-25 14:10:31 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-da55c8f0-663f-4d1e-ab8f-8c64d7695c03', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **questions** based ONLY on the provided text. Follow these rules:\n\n1. Generate **5–8 meaningful questions** per chunk.\n2. Questions must be **fully answerable from the text**.\n3. Avoid yes/no questions unless reasoning is involved.\n4. Do not use information **not present in the text**.\n5. Questions should be self-contained and understandable without context.\n6. Format your output as JSON:\n[\n  {"question": "Question 1"},\n  {"question": "Question 2"},\n  ...\n]\n'}, {'role': 'user', 'content': '\nGenerate 5–8 high-quality questions from the following chunk.\nOnly use in



[
  {"question": "What are the three main services offered by MindMap Digital according to the homepage?"},
  {"question": "How does Robotic Process Automation (RPA) help businesses reduce operational costs as described on the website?"},
  {"question": "What are the key features of MindMap Digital's Google Cloud & Google WorkSpace solutions?"},
  {"question": "What is the primary purpose of MindMap Digital's Generative AI service?"},
  {"question": "Which sections are directly accessible from the homepage navigation menu?"},
  {"question": "What benefits does MindMap Digital claim for its Robotic Process Automation solutions?"},
  {"question": "How does the website describe the impact of Google Cloud solutions on business growth?"},
  {"question": "What types of content and solutions can businesses generate using MindMap Digital's Generative AI service?"}
]




In [5]:
questions.root

[Question(question='What are the three main services offered by MindMap Digital according to the homepage?'),
 Question(question='How does Robotic Process Automation (RPA) help businesses reduce operational costs as described on the website?'),
 Question(question="What are the key features of MindMap Digital's Google Cloud & Google WorkSpace solutions?"),
 Question(question="What is the primary purpose of MindMap Digital's Generative AI service?"),
 Question(question='Which sections are directly accessible from the homepage navigation menu?'),
 Question(question='What benefits does MindMap Digital claim for its Robotic Process Automation solutions?'),
 Question(question='How does the website describe the impact of Google Cloud solutions on business growth?'),
 Question(question="What types of content and solutions can businesses generate using MindMap Digital's Generative AI service?")]

In [6]:
for i in questions.root:
    print(i.question)

What are the three main services offered by MindMap Digital according to the homepage?
How does Robotic Process Automation (RPA) help businesses reduce operational costs as described on the website?
What are the key features of MindMap Digital's Google Cloud & Google WorkSpace solutions?
What is the primary purpose of MindMap Digital's Generative AI service?
Which sections are directly accessible from the homepage navigation menu?
What benefits does MindMap Digital claim for its Robotic Process Automation solutions?
How does the website describe the impact of Google Cloud solutions on business growth?
What types of content and solutions can businesses generate using MindMap Digital's Generative AI service?


In [7]:
ans = obj.generate_answers(chunks[0], questions)

2026-02-25 14:10:33 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-9b6da7c2-0137-4efb-b803-0e795d73f1a6', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **answers** to the given questions based ONLY on the provided text. Follow these rules:\n\n1. Answers must be **grounded strictly in the text**.\n2. Do not include any external information.\n3. If the answer is **not present in the text**, return "NOT_ANSWERABLE_FROM_TEXT".\n4. Keep answers concise, clear, and structured.\n5. Return output in JSON:\n{\n  "answer": "Your answer here"\n}\n\nDo not reference the text itself (avoid phrases like "according to the text").\n'}, {'role': 'user', 'content': '\nAnswer the following question using only the text below. If the answer is not 



{"answer": "The three main services offered by MindMap Digital are Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI."}




2026-02-25 14:10:35 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 08:40:35 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'6000'), (b'x-ratelimit-remaining-requests', b'970'), (b'x-ratelimit-remaining-tokens', b'3351'), (b'x-ratelimit-reset-requests', b'43m12s'), (b'x-ratelimit-reset-tokens', b'26.49s'), (b'x-request-id', b'req_01kj9za1d2f7mts2521qx9h1zs'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'br'), (b'CF-RAY', b'9d35e60f49ad214e-DEL'), (b'alt-svc', b'h3=":443"; ma=86400')])
2026-02-25 14:10:35 - INFO - HTTP Request: POST https:/



{"answer": "Robotic Process Automation (RPA) helps businesses reduce operational costs by automating routine tasks and boosting productivity through scalable solutions."}




2026-02-25 14:10:35 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 08:40:35 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'6000'), (b'x-ratelimit-remaining-requests', b'969'), (b'x-ratelimit-remaining-tokens', b'3120'), (b'x-ratelimit-reset-requests', b'44m38.4s'), (b'x-ratelimit-reset-tokens', b'28.799s'), (b'x-request-id', b'req_01kj9za21bfkyrbm1b01f0314w'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'br'), (b'CF-RAY', b'9d35e61349a5214e-DEL'), (b'alt-svc', b'h3=":443"; ma=86400')])
2026-02-25 14:10:35 - INFO - HTTP Request: POST http

...



{"answer": "MindMap Digital claims that their Robotic Process Automation solutions automate routine tasks, boost productivity, and reduce operational costs."}




2026-02-25 14:10:37 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 08:40:37 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'6000'), (b'x-ratelimit-remaining-requests', b'965'), (b'x-ratelimit-remaining-tokens', b'2223'), (b'x-ratelimit-reset-requests', b'50m24s'), (b'x-ratelimit-reset-tokens', b'37.769s'), (b'x-request-id', b'req_01kj9za442f7tvtz1pejjnxjay'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'br'), (b'CF-RAY', b'9d35e620ad9a214e-DEL'), (b'alt-svc', b'h3=":443"; ma=86400')])
2026-02-25 14:10:37 - INFO - HTTP Request: POST https:



{"answer": "The website describes Google Cloud solutions as being designed for growth, collaboration, and data-driven success."}




2026-02-25 14:10:38 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 08:40:38 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'bom'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'6000'), (b'x-ratelimit-remaining-requests', b'964'), (b'x-ratelimit-remaining-tokens', b'1975'), (b'x-ratelimit-reset-requests', b'51m50.399s'), (b'x-ratelimit-reset-tokens', b'40.25s'), (b'x-request-id', b'req_01kj9za4npf7v9hngpjyskwjn6'), (b'via', b'1.1 google'), (b'cf-cache-status', b'DYNAMIC'), (b'Strict-Transport-Security', b'max-age=15552000'), (b'Content-Encoding', b'br'), (b'CF-RAY', b'9d35e6242c34214e-DEL'), (b'alt-svc', b'h3=":443"; ma=86400')])
2026-02-25 14:10:38 - INFO - HTTP Request: POST htt



{"answer": "unique content and solutions"}




In [8]:
ans

[AnswerOutput(answer='The three main services offered by MindMap Digital are Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI.'),
 AnswerOutput(answer='Robotic Process Automation (RPA) helps businesses reduce operational costs by automating routine tasks and boosting productivity through scalable solutions.'),
 AnswerOutput(answer="The key features of MindMap Digital's Google Cloud & Google WorkSpace solutions include being secure, scalable, innovative, and designed for growth, collaboration, and data-driven success."),
 AnswerOutput(answer="The primary purpose of MindMap Digital's Generative AI service is to unleash creativity and innovation by generating unique content and solutions."),
 AnswerOutput(answer='Home, About Us, Services'),
 AnswerOutput(answer='MindMap Digital claims that their Robotic Process Automation solutions automate routine tasks, boost productivity, and reduce operational costs.'),
 AnswerOutput(answer='The website describes Google C

In [9]:
# for ans in 

In [10]:
questions.root[0]

Question(question='What are the three main services offered by MindMap Digital according to the homepage?')

In [11]:
obj.judge_qa_pair(chunks[0], questions.root[0].question, ans[0].answer)

2026-02-25 14:10:38 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-ee01f0d9-b066-4a9b-8172-7d2bc174c641', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert evaluator for Question-Answer (QA) pairs generated from text chunks.\n\nYour task is to evaluate the quality of the QA pair based on the provided text chunk.\nFollow these rules:\n\n1.  **Relevance**: The question must be relevant to the text.\n2.  **Accuracy**: The answer must be accurate and fully supported by the text.\n3.  **Clarity**: Both question and answer should be clear and grammatical.\n4.  **Self-containment**: The question should be understandable without the text.\n\nOutput your evaluation strictly as JSON:\n{\n  "score": <float between 0.0 and 10.0>,\n  "reasoning": "<brief explanation of the score>"\n}\n'}, {'role': 'user', 'content': '\nEvaluate the following Question-Answer pair based on the pr



{
  "score": 10.0,
  "reasoning": "The question is relevant and self-contained, asking for the three main services listed on the homepage. The answer accurately reflects the services explicitly mentioned in the text (Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI) and is clearly phrased. All information is fully supported by the provided text, with no inaccuracies or ambiguities."
}




JudgeOutput(score=10.0, reasoning='The question is relevant and self-contained, asking for the three main services listed on the homepage. The answer accurately reflects the services explicitly mentioned in the text (Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI) and is clearly phrased. All information is fully supported by the provided text, with no inaccuracies or ambiguities.')

In [12]:
for question, answer in zip(questions.root, ans):
    print("-"*10)
    print(question.question)
    print(answer.answer)
    
    print("-"*10)


----------
What are the three main services offered by MindMap Digital according to the homepage?
The three main services offered by MindMap Digital are Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI.
----------
----------
How does Robotic Process Automation (RPA) help businesses reduce operational costs as described on the website?
Robotic Process Automation (RPA) helps businesses reduce operational costs by automating routine tasks and boosting productivity through scalable solutions.
----------
----------
What are the key features of MindMap Digital's Google Cloud & Google WorkSpace solutions?
The key features of MindMap Digital's Google Cloud & Google WorkSpace solutions include being secure, scalable, innovative, and designed for growth, collaboration, and data-driven success.
----------
----------
What is the primary purpose of MindMap Digital's Generative AI service?
The primary purpose of MindMap Digital's Generative AI service is to unleash crea

In [13]:
obj.run()

2026-02-25 14:10:39 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-fb8dfa7e-333e-4e38-ae48-3bde5c2c0284', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **questions** based ONLY on the provided text. Follow these rules:\n\n1. Generate **5–8 meaningful questions** per chunk.\n2. Questions must be **fully answerable from the text**.\n3. Avoid yes/no questions unless reasoning is involved.\n4. Do not use information **not present in the text**.\n5. Questions should be self-contained and understandable without context.\n6. Format your output as JSON:\n[\n  {"question": "Question 1"},\n  {"question": "Question 2"},\n  ...\n]\n'}, {'role': 'user', 'content': '\nGenerate 5–8 high-quality questions from the following chunk.\nOnly use in



[
  {"question": "What are the three main services offered by MindMap Digital according to the homepage?"},
  {"question": "What are the key benefits of Robotic Process Automation (RPA) as described on the website?"},
  {"question": "How does the website describe the purpose of Generative AI services?"},
  {"question": "What are the primary features of Google Cloud & Google WorkSpace solutions highlighted on the homepage?"},
  {"question": "Which sections are available in the navigation menu of the MindMap Digital website?"},
  {"question": "What operational advantages does the website claim for implementing RPA solutions?"},
  {"question": "What business outcomes are associated with Google Cloud solutions according to the description?"},
  {"question": "What creative capabilities does the Generative AI service promise to provide businesses?"}
]




2026-02-25 14:10:44 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-c25c3c6e-ff51-4782-9236-d4079df050dc', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **answers** to the given questions based ONLY on the provided text. Follow these rules:\n\n1. Answers must be **grounded strictly in the text**.\n2. Do not include any external information.\n3. If the answer is **not present in the text**, return "NOT_ANSWERABLE_FROM_TEXT".\n4. Keep answers concise, clear, and structured.\n5. Return output in JSON:\n{\n  "answer": "Your answer here"\n}\n\nDo not reference the text itself (avoid phrases like "according to the text").\n'}, {'role': 'user', 'content': '\nAnswer the following question using only the text below. If the answer is not 



{
  "answer": "The three main services offered by MindMap Digital are Robotic Process Automation, Google Cloud & Google WorkSpace, and Generative AI."
}




2026-02-25 14:10:50 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-2f3627f2-3a9a-4ad5-8cca-dba8468ca39a', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **answers** to the given questions based ONLY on the provided text. Follow these rules:\n\n1. Answers must be **grounded strictly in the text**.\n2. Do not include any external information.\n3. If the answer is **not present in the text**, return "NOT_ANSWERABLE_FROM_TEXT".\n4. Keep answers concise, clear, and structured.\n5. Return output in JSON:\n{\n  "answer": "Your answer here"\n}\n\nDo not reference the text itself (avoid phrases like "according to the text").\n'}, {'role': 'user', 'content': '\nAnswer the following question using only the text below. If the answer is not 

...

2026-02-25 14:14:30 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-2547438f-a414-465f-bfaa-b837ff37f2cb', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **answers** to the given questions based ONLY on the provided text. Follow these rules:\n\n1. Answers must be **grounded strictly in the text**.\n2. Do not include any external information.\n3. If the answer is **not present in the text**, return "NOT_ANSWERABLE_FROM_TEXT".\n4. Keep answers concise, clear, and structured.\n5. Return output in JSON:\n{\n  "answer": "Your answer here"\n}\n\nDo not reference the text itself (avoid phrases like "according to the text").\n'}, {'role': 'user', 'content': '\nAnswer the following question using only the text below. If the answer is not 



{"answer": "Redacto"}




2026-02-25 14:14:35 - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-356c1239-78e6-42a9-9fac-ec3c4e61339d', 'json_data': {'messages': [{'role': 'system', 'content': '\nYou are an expert dataset generation assistant for creating high-quality training data for instruction-tuned language models.\n\nYour task is to generate **answers** to the given questions based ONLY on the provided text. Follow these rules:\n\n1. Answers must be **grounded strictly in the text**.\n2. Do not include any external information.\n3. If the answer is **not present in the text**, return "NOT_ANSWERABLE_FROM_TEXT".\n4. Keep answers concise, clear, and structured.\n5. Return output in JSON:\n{\n  "answer": "Your answer here"\n}\n\nDo not reference the text itself (avoid phrases like "according to the text").\n'}, {'role': 'user', 'content': '\nAnswer the following question using only the text below. If the answer is not 



{"answer": "face match algorithm"}




KeyboardInterrupt: 